In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import sys
import os
sys.path.append(os.path.abspath("../Code"))
import vectorize
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../Data/raw/tcga_simple_train.csv')

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=33)

m_train,v = vectorize.vectorizacionBinaria(X_train)
m_test = v.transform(X_test)
m_val = v.transform(X_val)

le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_val_num = le.transform(y_val)
y_test_num = le.transform(y_test)

MAX_LEN = 200

le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_val_num = le.transform(y_val)
y_test_num = le.transform(y_test)


In [32]:
vocab_map = v.vocabulary_
index_mapping = {word: (idx + 1) for word, idx in vocab_map.items()}

In [33]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, Dense, LSTM, Embedding, Dropout

vocab_size = len(index_mapping) + 1

model = Sequential([
    
    Embedding(input_dim=vocab_size, output_dim=32, input_length=MAX_LEN),

    Bidirectional(LSTM(units=16, 
              activation='tanh', 
              dropout=0.5,
              recurrent_dropout=0,
              return_sequences=True)),
    
    Dropout(0.3),
    
    Bidirectional(LSTM(units=16, 
              activation='tanh', 
              dropout=0.5,
              recurrent_dropout=0)),
    
    Dense(4,activation='softmax')
])

model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])

c:\Users\tcspo\Desktop\Codigo_Clases\PLN1\practica\OncoNLP\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [34]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def texto_a_secuencia(serie_texto, mapping, max_len):
    sequencias = []
    for texto in serie_texto:

        seq = [mapping[w] for w in texto.split() if w in mapping]
        sequencias.append(seq)

    return pad_sequences(sequencias, maxlen=max_len)

X_train_rnn = texto_a_secuencia(X_train, index_mapping, MAX_LEN)
X_val_rnn = texto_a_secuencia(X_val, index_mapping, MAX_LEN)

In [35]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_rnn, y_train_num,
    epochs=20,
    batch_size=64,
    validation_data=(X_val_rnn, y_val_num)
)

Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 14s 125ms/step - accuracy: 0.3288 - loss: 1.3280 - val_accuracy: 0.3220 - val_loss: 1.3117
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.3482 - loss: 1.2957 - val_accuracy: 0.3245 - val_loss: 1.3067
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 104ms/step - accuracy: 0.3806 - loss: 1.2734 - val_accuracy: 0.3874 - val_loss: 1.2924
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 5s 104ms/step - accuracy: 0.4412 - loss: 1.2112 - val_accuracy: 0.3886 - val_loss: 1.2753
Epoch 5/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.5018 - loss: 1.1318 - val_accuracy: 0.4128 - val_loss: 1.2942
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.5570 - loss: 1.0462 - val_accuracy: 0.4225 - val_loss: 1.3129
Epoch 7/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.5967 - loss: 0.9823 - val_accuracy: 0.4237 - val_loss: 1.3218
Epoch 8/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 6s 107ms/step - accuracy: 0.6403 - loss: 0.9001 - val_accuracy: 0

In [36]:
X_test_rnn = texto_a_secuencia(X_test, index_mapping, MAX_LEN)

predicciones_prob = model.predict(X_test_rnn)
clases_predichas = np.argmax(predicciones_prob, axis=1)

print(classification_report(y_test_num,clases_predichas))

33/33 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step
              precision    recall  f1-score   support

           0       0.48      0.41      0.44       247
           1       0.45      0.54      0.49       354
           2       0.49      0.52      0.50       310
           3       0.40      0.24      0.30       121

    accuracy                           0.47      1032
   macro avg       0.46      0.43      0.43      1032
weighted avg       0.46      0.47      0.46      1032

